# 🎬 AI Movie Maker Colab（MiniMax H3-Turbo × autoip-mv-maker 整合改造版）

本筆記本將 **[autoip/autoip-mv-maker](https://github.com/autoip/autoip-mv-maker)** 的分鏡腳本架構與 **MiniMax H3-Turbo Colab (`1zdxzAm834HesHG2uu7GbM4Kv0kbKOLLu`)** 深度整合，專門解決以下四大核心需求：

1. **🛡️ 畫面連貫不跑掉（Visual Continuity & Anti-Drift）**
   - 支援 **多參考圖鎖定 (`Ref2VA` + `ref_image_size="max"`)**：固定 `<Picture 1>` 為主場景／180 度軸線基準圖，`<Picture 2..N>` 為固定角色設定圖。
   - 支援 **跨鏡自動接續尾幀 (`continuity_mode = "hybrid" | "chain_last_frame" | "anchor"`)**：同一場戲自動擷取上一鏡在 `tail_guard`（預設裁除最後 0.35 秒不穩定尾幀）之前的乾淨穩定畫面作為下一鏡的 `<Picture 1>`，徹底防止人物換臉、衣服變色或站位對調。
2. **🔊 嚴格單一聲軌（Single Audio Track Policy）**
   - 徹底解決「生成出來有兩個聲軌／雙重人聲疊加」的問題，提供三種互斥音軌模式，並在最終輸出自動透過 `ffprobe` 強制驗證 `len(audio_streams) == 1`：
     - `native_h3_audio`：只使用 MiniMax H3 原生單一語音（`<d>[Chinese] ...</d>`），自動抑制背景音樂與多重人聲。
     - `clip_voiceover`：使用外部配音／TTS 音檔（Prompt 自動關閉人物開口說話，並以 `-map 0:v:0 -map 1:a:0` 完全替換掉 H3 原生音軌，絕不混音疊加）。
     - `master_audio`：MV 或整支總旁白音軌模式（所有生成片段先以 `-an` 徹底拔除音軌，最後只掛載唯一一條主音軌）。
3. **📋 分鏡腳本直接驅動生成（Storyboard-Driven）**
   - 直接貼上 **Markdown 分鏡表格**、上傳 **`scenes.json` / `shotlist.json`**，或載入 `autoip-mv-maker` 的 **`project.py`**，自動編譯防跑掉提示詞與鏡頭工作流。
4. **📝 自動對齊並燒錄繁體中文字幕（Auto Subtitles）**
   - 自動依據分鏡台詞與實際片長切分繁體中文字句，支援 `faster-whisper` 語音時間軸強制對齊（保留分鏡正確繁體字、不產生同音錯字），同時輸出 `.zh-TW.srt`、`.ass`、無字幕版 `_clean.mp4` 與燒錄字幕版 `_subtitled.mp4`。

In [ ]:
#@title Step 1. 檢查 GPU、掛載 Google Drive 模型快取、安裝 AI-Movie (`mvkit`) 與 ComfyUI 環境
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI_MiniMax_H3" #@param {type:"string"}
INSTALL_FASTER_WHISPER = True #@param {type:"boolean"}
REPO_URL = "https://github.com/a017749-chen/AI-movie.git" #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        Path(DRIVE_ROOT, "models").mkdir(parents=True, exist_ok=True)
        Path(DRIVE_ROOT, "outputs").mkdir(parents=True, exist_ok=True)
        print(f"[OK] Google Drive 已掛載，模型與成品快取目錄：{DRIVE_ROOT}")
    except Exception as exc:
        print(f"[WARN] 未掛載 Google Drive（{exc}），將使用 Colab 本機磁碟。")

# 1. 安裝系統字型 (Noto Sans CJK TC 供繁體中文字幕燒錄) 與 ffmpeg
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-noto-cjk", "aria2"], check=False)

# 2. Clone 或更新 AI-movie 專案
repo_dir = Path("/content/AI-movie")
if not repo_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# 3. 安裝 ComfyUI (MiniMax H3 分支) 與必要依賴
comfy_dir = Path("/content/ComfyUI")
if not comfy_dir.exists():
    subprocess.run(
        ["git", "clone", "-b", "minimax/h3", "--depth", "1", "https://github.com/MiniMax-AI/ComfyUI.git", str(comfy_dir)],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(comfy_dir / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pillow", "librosa", "soundfile"], check=True)

if INSTALL_FASTER_WHISPER:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=False)

print("[READY] AI-Movie (mvkit) + ComfyUI MiniMax H3 環境準備完成！")

In [ ]:
#@title Step 2. 啟動背景 ComfyUI 伺服器並準備參考圖／音檔目錄
import subprocess
import time
import urllib.request
from pathlib import Path

WORKSPACE_DIR = Path("/content/workspace_aimovie")
REFS_DIR = WORKSPACE_DIR / "refs"
AUDIO_DIR = WORKSPACE_DIR / "audio"
OUTPUT_DIR = WORKSPACE_DIR / "outputs"
for d in (WORKSPACE_DIR, REFS_DIR, AUDIO_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def is_comfy_alive(url: str = "http://127.0.0.1:8188/system_stats") -> bool:
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            return resp.status == 200
    except Exception:
        return False

if not is_comfy_alive():
    log_path = WORKSPACE_DIR / "comfyui.log"
    log_file = open(log_path, "w", encoding="utf-8")
    subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    print("正在啟動 ComfyUI 伺服器，請稍候...")
    for _ in range(90):
        if is_comfy_alive():
            break
        time.sleep(2)

if is_comfy_alive():
    print("[OK] ComfyUI 伺服器已就緒 (http://127.0.0.1:8188)")
else:
    raise RuntimeError("ComfyUI 啟動超時，請檢查 /content/workspace_aimovie/comfyui.log")

In [ ]:
#@title Step 3. 上傳參考圖（場景基準圖 `<Picture 1>`、角色圖 `<Picture 2..N>`）或外部音檔（選填）
UPLOAD_REFERENCE_IMAGES = False #@param {type:"boolean"}
UPLOAD_AUDIO_FILES = False #@param {type:"boolean"}

from pathlib import Path
from PIL import Image, ImageDraw

if UPLOAD_REFERENCE_IMAGES:
    from google.colab import files
    print("請選擇要上傳的參考圖片（例如 scene_anchor.png, char_learner.png 等）：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = REFS_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存參考圖：{dst}")

if UPLOAD_AUDIO_FILES:
    from google.colab import files
    print("請選擇要上傳的單一主音軌或分段旁白 WAV/MP3：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = AUDIO_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存音檔：{dst}")

# 若尚未上傳參考圖，建立一張預設穩定構圖基準圖供測試
default_anchor = REFS_DIR / "scene_anchor.png"
if not any(REFS_DIR.glob("*.png")) and not any(REFS_DIR.glob("*.jpg")):
    img = Image.new("RGB", (1280, 720), color=(34, 45, 62))
    draw = ImageDraw.Draw(img)
    draw.rectangle([80, 80, 1200, 640], outline=(180, 200, 220), width=3)
    img.save(default_anchor)
    print(f"[INFO] 尚未上傳圖片，已自動建立預設場景基準圖：{default_anchor}")
else:
    imgs = sorted(list(REFS_DIR.glob("*.png")) + list(REFS_DIR.glob("*.jpg")) + list(REFS_DIR.glob("*.webp")))
    default_anchor = imgs[0]
    print(f"[OK] 目前可用參考圖：{[p.name for p in imgs]}")

In [ ]:
#@title Step 4. 設定分鏡腳本、畫面連貫鎖定模式與「單一聲軌」策略
PROJECT_TITLE = "AI_Movie_Output" #@param {type:"string"}
STORYBOARD_SOURCE_TYPE = "markdown_inline" #@param ["markdown_inline", "file_path", "upload_file"]
STORYBOARD_FILE_PATH = "/content/AI-movie/examples/storyboard_template.md" #@param {type:"string"}

#@markdown ---
#@markdown ### 🔊 單一聲軌設定（保證最終影片只有唯一一條音軌，絕不雙聲軌疊加）
AUDIO_MODE = "native_h3_audio" #@param ["native_h3_audio", "clip_voiceover", "master_audio"]
MASTER_AUDIO_PATH = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🛡️ 畫面防跑掉與連貫性設定（Visual Continuity & Anti-Drift）
INFERENCE_MODE = "reference" #@param ["reference", "first_frame", "text_only"]
CONTINUITY_MODE = "hybrid" #@param ["hybrid", "chain_last_frame", "anchor"]
REF_IMAGE_SIZE = "max" #@param ["max", "match"]
TAIL_GUARD_SEC = 0.35 #@param {type:"number"}
ASPECT_RATIO = "16:9" #@param ["16:9", "9:16", "1:1"]
FPS = 24 #@param {type:"integer"}
STEPS = 8 #@param {type:"integer"}
CFG = 4.0 #@param {type:"number"}

# 可直接在此貼上你的 Markdown 分鏡表格！
MARKDOWN_STORYBOARD = '''
| 鏡號 | 秒數 | 場景 | 畫面與動作描述 | 運鏡 | 旁白 / 台詞 | 字幕 | 參考圖 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 01_opening | 0-8s | station | 明亮教學空間，左側年輕女性學員手持紀錄板專注報告，右側資深男導師微笑點頭傾聽，雙人站位與視線軸線穩定。 | 固定雙人中景，維持 180 度軸線 | 大家好，今天我們用一分鐘的時間，帶大家掌握結構化交班的四個核心步驟。 | 今天帶大家掌握結構化交班的四個核心步驟 | scene_anchor.png |
| 02_situation | 8-18s | station | 延續上一鏡相同場景與站位，左側學員翻開紀錄表指出重點欄位，右側導師看著紀錄表並給予肯定點頭。 | 極緩慢微推中近景，嚴禁跳軸 | 第一步是說明目前狀況與關鍵背景，讓接手團隊在最短時間內抓到重點。 | 第一步：說明目前狀況與關鍵背景，讓團隊快速抓到重點 | scene_anchor.png |
| 03_assessment | 18-28s | station | 左側學員條理分明地比出兩個重點手勢，表情沉穩自信；右側導師在旁紀錄並微笑回應。 | 固定雙人中景，背景與光影一致 | 第二步是提出精準評估與具體建議，明確告訴團隊接下來需要配合的處置方向。 | 第二步：提出精準評估與具體建議，明確說明處置方向 | scene_anchor.png |
| 04_closing | 28-36s | station | 雙人完成交班，學員合上紀錄板，導師比出讚許手勢，兩人同時面向鏡頭自然微笑收尾。 | 固定穩定中景，平穩收尾 | 掌握結構、穩定表達，就能讓每一次團隊溝通既安全又高效率！ | 掌握結構、穩定表達，讓團隊溝通既安全又高效率！ | scene_anchor.png |
'''

from mvkit.script_loader import (
    compile_shot_prompt,
    load_storyboard,
    load_storyboard_from_markdown,
    resolve_shot_references,
)

if STORYBOARD_SOURCE_TYPE == "upload_file":
    from google.colab import files
    uploaded = files.upload()
    up_name = next(iter(uploaded.keys()))
    up_path = WORKSPACE_DIR / up_name
    up_path.write_bytes(uploaded[up_name])
    STORYBOARD_FILE_PATH = str(up_path)
    STORYBOARD_SOURCE_TYPE = "file_path"

if STORYBOARD_SOURCE_TYPE == "markdown_inline":
    project = load_storyboard_from_markdown(
        MARKDOWN_STORYBOARD,
        title=PROJECT_TITLE,
        audio_mode=AUDIO_MODE,
        continuity_mode=CONTINUITY_MODE,
        default_reference_image=str(default_anchor),
        base_dir=REFS_DIR,
    )
else:
    project = load_storyboard(
        STORYBOARD_FILE_PATH,
        audio_mode=AUDIO_MODE,
        continuity_mode=CONTINUITY_MODE,
        default_reference_image=str(default_anchor),
    )
    project.title = PROJECT_TITLE

project.inference_mode = INFERENCE_MODE
project.ref_image_size = REF_IMAGE_SIZE
project.tail_guard_sec = float(TAIL_GUARD_SEC)
project.aspect_ratio = ASPECT_RATIO
project.fps = int(FPS)
project.steps = int(STEPS)
project.cfg = float(CFG)
if MASTER_AUDIO_PATH:
    project.master_audio_path = MASTER_AUDIO_PATH

for shot in project.shots:
    shot.reference_images = resolve_shot_references(project, shot, base_dir=REFS_DIR)
    compile_shot_prompt(project, shot)

print(f"✅ 已載入分鏡專案：{project.title}（共 {len(project.shots)} 個鏡頭）")
print(f"   - 單一聲軌模式 (audio_mode)     : {project.audio_mode}")
print(f"   - 畫面連貫模式 (continuity_mode): {project.continuity_mode} (tail_guard={project.tail_guard_sec}s)")
print(f"   - 推論模式 (inference_mode)     : {project.inference_mode} (ref_image_size={project.ref_image_size})")
print("\n--- 第一鏡編譯後之防跑掉提示詞預覽 ---")
if project.shots:
    print(project.shots[0].compiled_h3_prompt)

In [ ]:
#@title Step 5. 依分鏡腳本批次生成影片片段（含自動跨鏡尾幀接續與斷點續傳）
RESUME_FROM_CHECKPOINT = True #@param {type:"boolean"}

from mvkit.h3_colab_runner import H3ColabRunner

drive_models = f"{DRIVE_ROOT}/models" if MOUNT_GOOGLE_DRIVE else None
runner = H3ColabRunner(
    comfy_dir="/content/ComfyUI",
    server_url="http://127.0.0.1:8188",
    drive_models_dir=drive_models,
)

raw_clips = runner.run_storyboard(
    project,
    OUTPUT_DIR / project.title,
    base_dir=REFS_DIR,
    resume=RESUME_FROM_CHECKPOINT,
)
print(f"🎉 全部 {len(raw_clips)} 個分鏡片段生成完畢！")
for p in raw_clips:
    print(" -", p)

In [ ]:
#@title Step 6. 單一聲軌合成、自動時間軸對齊與繁體中文字幕燒錄、品質檢視表與下載
USE_ASR_FORCED_ALIGNMENT = True #@param {type:"boolean"}
BURN_IN_SUBTITLES = True #@param {type:"boolean"}
AUTO_DOWNLOAD_FINAL_MP4 = False #@param {type:"boolean"}

import shutil
from IPython.display import HTML, Image as IPyImage, display
from base64 import b64encode
from mvkit.colab_pipeline import assemble_movie

manifest = assemble_movie(
    project,
    raw_clips,
    OUTPUT_DIR / project.title,
    base_dir=AUDIO_DIR,
    use_asr_alignment=USE_ASR_FORCED_ALIGNMENT,
    burn_subtitles=BURN_IN_SUBTITLES,
)

print("============================================================")
print("✅ 成品輸出與「單一聲軌」驗證通過！")
print(f"   - 乾淨無字幕版 MP4 : {manifest['clean_mp4']}")
print(f"   - 燒錄繁中字幕 MP4 : {manifest['subtitled_mp4']}")
print(f"   - SRT 字幕檔       : {manifest['srt_path']}")
print(f"   - ASS 字幕檔       : {manifest['ass_path']}")
print(f"   - 連貫性檢視圖表   : {manifest['review_sheet']}")
print(f"   - 音軌驗證結果     : 1 條視訊軌 + 1 條音訊軌 ({manifest['clean_stream_verification']['audio_stream']['codec_name']}, {manifest['clean_stream_verification']['audio_stream']['sample_rate']} Hz)")
print("============================================================")

if MOUNT_GOOGLE_DRIVE and Path(DRIVE_ROOT).exists():
    drive_out = Path(DRIVE_ROOT) / "outputs" / project.title
    drive_out.mkdir(parents=True, exist_ok=True)
    for key in ("clean_mp4", "subtitled_mp4", "srt_path", "ass_path", "review_sheet"):
        val = manifest.get(key)
        if val and Path(val).exists():
            shutil.copy2(val, drive_out / Path(val).name)
    print(f"📂 已同步備份至 Google Drive：{drive_out}")

if manifest.get("review_sheet") and Path(manifest["review_sheet"]).exists():
    display(IPyImage(filename=manifest["review_sheet"]))

final_video = manifest.get("subtitled_mp4") or manifest["clean_mp4"]
if Path(final_video).exists() and Path(final_video).stat().st_size < 45 * 1024 * 1024:
    mp4_bytes = Path(final_video).read_bytes()
    b64 = b64encode(mp4_bytes).decode("ascii")
    display(HTML(f'<video width="840" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

if AUTO_DOWNLOAD_FINAL_MP4:
    from google.colab import files
    files.download(final_video)
    if manifest.get("srt_path"):
        files.download(manifest["srt_path"])